## Evaluation

In [1]:
from anatolian_sam.eval_utils import (
    hz_to_cents,
    extract_f0_pyin,
    plot_and_evaluate_3way,
    compute_emd_metrics,
)
from pathlib import Path

/teamspace/studios/this_studio/anatolian-SAM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import json
import os

import librosa
from tqdm import tqdm


def run_bulk_evaluation(
    val_metadata,
    gt_dir,
    separated_dir,
    eval_dir,
    domain_label,
    zs_prefix="zeroshot_",
    lora_prefix="lora_",
):
    """
    Evaluates a directory of test samples.
    """
    # load metadata
    metadata = []
    with open(val_metadata, "r") as f:
        for line in f:
            if line.strip():
                metadata.append(json.loads(line))

    gt_dir = Path(gt_dir)
    separated_dir = Path(separated_dir)

    # Create an output directory for the plots so they don't flood your notebook
    plot_dir = Path(eval_dir) / "plots" / domain_label
    plot_dir.mkdir(parents=True, exist_ok=True)

    results = []

    # ground truth audio files
    audio_filenames = []
    for stem in metadata:
        audio_filenames.append(
            {
                "filename": os.path.split(stem["target_path"])[-1],
                "instrument": stem["selected_instrument_info"]["target"]["instrument"],
            }
        )

    for audio_file in tqdm(audio_filenames, desc=f"Evaluating {domain_label}"):
        filename = audio_file["filename"]
        instrument = audio_file["instrument"]

        try:
            # Load audio
            y_gt, sr = librosa.load(gt_dir / filename, sr=44100)
            y_zs, _ = librosa.load(separated_dir / f"{zs_prefix}{filename}.wav", sr=sr)
            y_lora, _ = librosa.load(
                separated_dir / f"{lora_prefix}{filename}.wav", sr=sr
            )

            # Compute metrics
            emd_zs, emd_lora, cents_tuple = compute_emd_metrics(y_gt, y_zs, y_lora, sr)

            # Store results as a dict to build the DataFrame efficiently later
            results.append(
                {
                    "filename": filename,
                    "instrument": instrument,
                    "domain": domain_label,
                    "emd_zero_shot": emd_zs,
                    "emd_lora": emd_lora,
                }
            )

            # Optional: Save the plot to disk instead of rendering it inline
            cents_gt, cents_zs, cents_lora = cents_tuple
            plot_and_evaluate_3way(
                cents_gt, cents_zs, cents_lora, filename, plot_dir, save=True
            )

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # Directly convert the list of dicts to a DataFrame
    return pd.DataFrame(results)

In [8]:
import pandas as pd


df_turkish = run_bulk_evaluation(
    val_metadata="../data/val_metadata_tr.jsonl",
    gt_dir="/teamspace/studios/turkish-music/anatolian-SAM/data/mixed",
    separated_dir="../evaluations/audio_results",
    eval_dir="../evaluations",
    domain_label="in_domain_turkish",
)

df_western = run_bulk_evaluation(
    val_metadata="../data/val_metadata_ww.jsonl",
    gt_dir="/teamspace/studios/turkish-music/anatolian-SAM/data/mixed/ww",
    separated_dir="../evaluations/audio_results_ww",
    eval_dir="../evaluations",
    domain_label="out_of_domain_western",
)

df_final = pd.concat([df_turkish, df_western], ignore_index=True)

Evaluating out_of_domain_western: 100%|██████████| 144/144 [12:15<00:00,  5.11s/it]


In [9]:
df_final.to_csv("../evaluations/emd_results.csv", index=False)

In [10]:
# Calculate the mean EMD for both domains and both models
matrix_summary = df_final.groupby("domain")[["emd_zero_shot", "emd_lora"]].mean()

print("\n--- Final 2x2 Experimental Matrix (Mean EMD in Cents) ---")
print(matrix_summary)


--- Final 2x2 Experimental Matrix (Mean EMD in Cents) ---
                       emd_zero_shot    emd_lora
domain                                          
in_domain_turkish         171.627137  895.781482
out_of_domain_western     604.452168  659.723333
